# UR5 equations of motion — three forward-dynamics pipelines

Textbook-style comparison of how minilink evaluates UR5 joint accelerations $\ddot q$ from

$$
H(q)\,\ddot q + C(q,\dot q)\,\dot q + d(q,\dot q) + g(q) = \tau.
$$

Standalone notebook: only **minilink** (+ NumPy / SymPy / JAX). Helpers live in a collapsible cell below.

**§0** free-motion meshcat · **§1–4** math of each pipeline · **§5** one-shot FD · **§6** small batch · **§7** short EoM integration.

| Pipeline | Idea | minilink entry |
| --- | --- | --- |
| **RNEA–$H$** | RNEA bias + explicit inertia solve | `UR5Manipulator.forward_dynamics_rnea_h` |
| **ABA** | Articulated-body algorithm ($O(n)$ spatial) | `UR5Manipulator.forward_dynamics` (catalog default) |
| **Symbolic Lagrange** | Derive $H,C,g$ once; lambdify to JAX | `minilink.symbolic` → `to_minilink(backend="jax")` |

**Inverse dynamics:** catalog default is spatial **RNEA** (`inverse_dynamics`); matrix form is `inverse_dynamics_matrix`.


In [2]:
import sys
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

# Local conda: minilink already installed. Colab: clone + path + meshcat.
import importlib.util
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

_OPTIMIZER_METHOD = (
    "ipopt" if importlib.util.find_spec("cyipopt") is not None else "scipy_slsqp"
)


from minilink.dynamics.catalog.manipulators.ur5 import UR5Manipulator
from minilink.simulation.simulator import Simulator


Cloning into 'minilink'...
remote: Enumerating objects: 9622, done.
remote: Counting objects: 100% (1082/1082), done.
remote: Compressing objects: 100% (377/377), done.
remote: Total 9622 (delta 849), reused 765 (delta 698), pack-reused 8540 (from 3)
Receiving objects: 100% (9622/9622), 144.69 MiB | 21.25 MiB/s, done.
Resolving deltas: 100% (6642/6642), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 77.6 MB/s eta 0:00:00


<details>
<summary><b>Notebook helpers</b> — run once (click to expand / read)</summary>

Self-contained utilities:

- **Symbolic UR5** — DH chain from catalog params → Lagrange derive → one JAX `to_minilink` export (cached).
- **Single-step batch** — a few random $(q,\dot q,\tau)$; ABA / Symbolic vs RNEA–$H$ (`max |Δq̈|` + median ms).
- **EoM integration** — short RK4 grid for RNEA–$H$ vs ABA (symbolic omitted here — each symbolic FD call is expensive); time *compile* / *JIT* / *integrate*.

Defaults: `5` samples, short $t_f=0.05\,\mathrm{s}$ integration.

</details>


In [3]:
# --- Notebook helpers (self-contained) ---
import os
import time
from dataclasses import dataclass, field

DEFAULT_SEED = 0
DEFAULT_N_SAMPLES = 5  # small reproducible batch
DEFAULT_N_TIMING = 1
DEFAULT_INTEGRATION_TF = 5.0
DEFAULT_INTEGRATION_DT = 0.05
DEFAULT_INTEGRATION_BACKEND = "jax"
DEFAULT_INTEGRATION_SOLVER = "rk4_fixedsteps"

_SYM_SYS = None
_PARAM_MAP = None
_PLANTS = {}


def catalog_params_no_damping():
    """Match symbolic Lagrange (no viscous d) to catalog spatial params."""
    params = dict(UR5Manipulator().params)
    params["damping"] = np.zeros(6)
    return params


def derive_symbolic_ur5(*, verbose=False):
    """Lagrange derive once; cache SymPy H, C, g."""
    global _SYM_SYS, _PARAM_MAP
    if _SYM_SYS is not None:
        return _SYM_SYS

    from minilink.symbolic.mechanics.model import MechanicalModel

    catalog = UR5Manipulator()
    params = catalog.params
    model = MechanicalModel("UR5Symbolic")
    coords = model.coordinates("q1 q2 q3 q4 q5 q6")
    g_sym = model.parameters("g")

    dh_table, link_properties = [], []
    for i in range(catalog.dof):
        inertia = params["inertia"][i]
        dh_table.append(
            {
                "theta": coords[i],
                "d": float(params["d"][i]),
                "a": float(params["a"][i]),
                "alpha": float(params["alpha"][i]),
            }
        )
        link_properties.append(
            {
                "mass": float(params["mass"][i]),
                "inertia": {
                    "Ixx": float(inertia[0, 0]),
                    "Iyy": float(inertia[1, 1]),
                    "Izz": float(inertia[2, 2]),
                },
                "com_offset": {
                    "x": float(params["com"][i, 0]),
                    "y": float(params["com"][i, 1]),
                    "z": float(params["com"][i, 2]),
                },
            }
        )

    model.add_dh_chain(dh_table, link_properties)
    model.add_gravity(-g_sym * model.N.z)

    if verbose:
        print("Deriving symbolic UR5 EoM (Lagrange, simplify=False)...")
    t0 = time.perf_counter()
    sym_sys = model.derive(method="lagrange", simplify=False)
    if verbose:
        print(f"  derive: {time.perf_counter() - t0:.1f} s")

    _SYM_SYS = sym_sys
    _PARAM_MAP = {g_sym: float(params["gravity"])}
    return sym_sys


def build_symbolic_ur5(*, backend=DEFAULT_INTEGRATION_BACKEND, verbose=False):
    """One SymPy → minilink export (default JAX). Cached per backend."""
    global _PLANTS
    if backend in _PLANTS:
        return _PLANTS[backend]
    sym_sys = derive_symbolic_ur5(verbose=verbose)
    if verbose:
        print(f"Exporting to minilink (backend={backend!r})...")
    t0 = time.perf_counter()
    plant = sym_sys.to_minilink(parameters=_PARAM_MAP, backend=backend)
    if verbose:
        print(f"  export: {time.perf_counter() - t0:.1f} s")
    _PLANTS[backend] = plant
    return plant


def _entry_term_count(expr):
    import sympy as sp

    if expr == 0:
        return 0
    return len(sp.Add.make_args(expr))


def symbolic_matrix_complexity(matrix):
    """Top-level term counts + count_ops for a SymPy matrix."""
    import sympy as sp

    M = sp.Matrix(matrix)
    rows, cols = M.shape
    term_counts = np.zeros((rows, cols), dtype=int)
    op_counts = np.zeros((rows, cols), dtype=int)
    for i in range(rows):
        for j in range(cols):
            term_counts[i, j] = _entry_term_count(M[i, j])
            op_counts[i, j] = int(sp.count_ops(M[i, j]))
    i_max, j_max = np.unravel_index(int(np.argmax(op_counts)), op_counts.shape)
    return {
        "shape": (rows, cols),
        "term_counts": term_counts,
        "op_counts": op_counts,
        "total_terms": int(term_counts.sum()),
        "max_terms": int(term_counts.max()),
        "mean_terms": float(term_counts.mean()),
        "total_ops": int(op_counts.sum()),
        "max_ops": int(op_counts.max()),
        "densest_entry": (int(i_max), int(j_max)),
        "n_zero": int(np.count_nonzero(term_counts == 0)),
    }


def format_complexity_table(stats):
    counts = stats["term_counts"]
    rows, cols = counts.shape
    header = "|  | " + " | ".join(f"j={j + 1}" for j in range(cols)) + " |"
    sep = "| --- | " + " | ".join("---" for _ in range(cols)) + " |"
    lines = [header, sep]
    for i in range(rows):
        cells = " | ".join(str(int(counts[i, j])) for j in range(cols))
        lines.append(f"| i={i + 1} | {cells} |")
    return "\n".join(lines)


@dataclass
class MethodStats:
    name: str
    per_sample_max: np.ndarray

    @property
    def max(self):
        return float(np.max(self.per_sample_max))


@dataclass
class ComparisonResult:
    seed: int
    n_samples: int
    methods: dict = field(default_factory=dict)
    timing_ms: dict = field(default_factory=dict)


def build_evaluation_batch(seed=DEFAULT_SEED, n_samples=DEFAULT_N_SAMPLES):
    """A few random (q, v, u) samples."""
    rng = np.random.default_rng(seed)
    configs = []
    for _ in range(n_samples):
        configs.append(
            (
                rng.uniform(-1.0, 1.0, 6),
                rng.uniform(-1.0, 1.0, 6),
                rng.uniform(-5.0, 5.0, 6),
            )
        )
    return configs


def forward_dynamics_rnea_h(arm, q, v, u, params):
    return arm.forward_dynamics_rnea_h(q, v, u, params=params)


def forward_dynamics_aba(arm, q, v, u, params):
    return arm.forward_dynamics(q, v, u, params=params)


def forward_dynamics_symbolic(plant, q, v, u):
    return plant.forward_dynamics(q, v, u)


def _benchmark_median(func, configs, n_repeat, *args):
    times = []
    n_cfg = len(configs)
    for k in range(n_repeat):
        q, v, u = configs[k % n_cfg]
        t0 = time.perf_counter()
        func(q, v, u, *args)
        times.append(time.perf_counter() - t0)
    return float(np.median(times))


def comprehensive_comparison(
    arm,
    symbolic_plant,
    configs,
    *,
    params,
    seed=DEFAULT_SEED,
    n_timing=DEFAULT_N_TIMING,
):
    """ABA / Symbolic vs RNEA–H: max |Δq̈| and median call time."""
    method_errors = {"ABA": []}
    if symbolic_plant is not None:
        method_errors["Symbolic"] = []

    for q, v, u in configs:
        qdd_ref = forward_dynamics_rnea_h(arm, q, v, u, params)
        qdd_aba = forward_dynamics_aba(arm, q, v, u, params)
        method_errors["ABA"].append(float(np.max(np.abs(qdd_ref - qdd_aba))))
        if symbolic_plant is not None:
            qdd_sym = forward_dynamics_symbolic(symbolic_plant, q, v, u)
            method_errors["Symbolic"].append(float(np.max(np.abs(qdd_ref - qdd_sym))))

    # one warm call each, then median timing
    q0, v0, u0 = configs[0]
    forward_dynamics_rnea_h(arm, q0, v0, u0, params)
    forward_dynamics_aba(arm, q0, v0, u0, params)
    if symbolic_plant is not None:
        forward_dynamics_symbolic(symbolic_plant, q0, v0, u0)

    timing_ms = {
        "RNEA-H": 1e3
        * _benchmark_median(
            lambda q, v, u, p: forward_dynamics_rnea_h(arm, q, v, u, p),
            configs,
            n_timing,
            params,
        ),
        "ABA": 1e3
        * _benchmark_median(
            lambda q, v, u, p: forward_dynamics_aba(arm, q, v, u, p),
            configs,
            n_timing,
            params,
        ),
    }
    if symbolic_plant is not None:
        timing_ms["Symbolic"] = 1e3 * _benchmark_median(
            lambda q, v, u: forward_dynamics_symbolic(symbolic_plant, q, v, u),
            configs,
            n_timing,
        )

    methods = {
        name: MethodStats(name=name, per_sample_max=np.asarray(errs))
        for name, errs in method_errors.items()
    }
    return ComparisonResult(
        seed=seed,
        n_samples=len(configs),
        methods=methods,
        timing_ms=timing_ms,
    )


def accuracy_summary(result):
    rows = [
        {
            "method": "RNEA-H (ref)",
            "max_abs_dqdd": 0.0,
            "median_ms": result.timing_ms["RNEA-H"],
        }
    ]
    for name, stats in result.methods.items():
        rows.append(
            {
                "method": name,
                "max_abs_dqdd": stats.max,
                "median_ms": result.timing_ms[name],
            }
        )
    return rows


def aba_speedup(result):
    return result.timing_ms["RNEA-H"] / max(result.timing_ms["ABA"], 1e-12)


class UR5ManipulatorRNEA(UR5Manipulator):
    """Same plant, but f uses RNEA–H forward dynamics (integration reference)."""

    def forward_dynamics(self, q, v, u, t=0.0, params=None):
        return self.forward_dynamics_rnea_h(q, v, u, t, params)


@dataclass
class IntegrationTiming:
    method: str
    compile_ms: float
    jit_warmup_ms: float
    integrate_ms: float
    backend: str = DEFAULT_INTEGRATION_BACKEND


@dataclass
class IntegrationComparisonResult:
    backend: str
    tf: float
    dt: float
    n_steps: int
    timings: list = field(default_factory=list)
    max_state_error: dict = field(default_factory=dict)


def timed_integration(sys, *, method, compile_backend, tf, dt, solver):
    """compile → first solve (JIT) → second solve (EoM integration only)."""
    if compile_backend == "jax":
        from minilink.core.backends import configure_jax

        configure_jax(enable_x64=True)

    kwargs = dict(x0=sys.x0, t0=0.0, tf=tf, dt=dt, solver=solver, verbose=False)
    t0 = time.perf_counter()
    sim = Simulator(sys, compile_backend=compile_backend, **kwargs)
    compile_ms = 1e3 * (time.perf_counter() - t0)

    t0 = time.perf_counter()
    sim.solve()
    jit_warmup_ms = 1e3 * (time.perf_counter() - t0)

    t0 = time.perf_counter()
    traj = sim.solve()
    integrate_ms = 1e3 * (time.perf_counter() - t0)

    return traj, IntegrationTiming(
        method=method,
        compile_ms=compile_ms,
        jit_warmup_ms=jit_warmup_ms,
        integrate_ms=integrate_ms,
        backend=compile_backend,
    )


def eom_integration_comparison(
    params,
    *,
    symbolic_plant=None,
    compile_backend=DEFAULT_INTEGRATION_BACKEND,
    tf=DEFAULT_INTEGRATION_TF,
    dt=DEFAULT_INTEGRATION_DT,
    solver=DEFAULT_INTEGRATION_SOLVER,
):
    """Integrate RNEA–H / ABA / optional symbolic on a short EoM grid."""
    q0 = np.array([0.0, -np.pi / 2 + 0.2, 0.0, -np.pi / 2, 0.0, 0.0])
    v0 = np.array([0.2, -0.1, 0.15, -0.05, 0.08, -0.03])

    def _prep(plant):
        plant.params = dict(params)
        plant.x0 = plant.q2x(q0, v0)
        plant.inputs["u"].nominal_value = np.zeros(plant.m)
        return plant

    result = IntegrationComparisonResult(
        backend=compile_backend,
        tf=tf,
        dt=dt,
        n_steps=int(round(tf / dt)) + 1,
    )

    ref_traj, ref_timing = timed_integration(
        _prep(UR5ManipulatorRNEA()),
        method="RNEA-H",
        compile_backend=compile_backend,
        tf=tf,
        dt=dt,
        solver=solver,
    )
    result.timings.append(ref_timing)

    aba_traj, aba_timing = timed_integration(
        _prep(UR5Manipulator()),
        method="ABA",
        compile_backend=compile_backend,
        tf=tf,
        dt=dt,
        solver=solver,
    )
    result.timings.append(aba_timing)
    result.max_state_error["ABA"] = float(np.max(np.abs(ref_traj.x - aba_traj.x)))

    if symbolic_plant is not None:
        sym_traj, sym_timing = timed_integration(
            _prep(symbolic_plant),
            method="Symbolic",
            compile_backend=compile_backend,
            tf=tf,
            dt=dt,
            solver=solver,
        )
        result.timings.append(sym_timing)
        result.max_state_error["Symbolic"] = float(
            np.max(np.abs(ref_traj.x - sym_traj.x))
        )

    return result, ref_traj


def _fmt_cell(value):
    if isinstance(value, float):
        if value == 0.0:
            return "0"
        if abs(value) >= 100.0 or abs(value) < 0.01:
            return f"{value:.2e}"
        return f"{value:.2f}"
    return str(value)


def rows_to_markdown(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = [
        "| " + " | ".join(_fmt_cell(row[c]) for c in columns) + " |" for row in rows
    ]
    return chr(10).join([header, sep, *body])


def integration_timing_rows(result):
    return [
        {
            "method": t.method,
            "compile_ms": t.compile_ms,
            "jit_warmup_ms": t.jit_warmup_ms,
            "integrate_ms": t.integrate_ms,
            "backend": t.backend,
        }
        for t in result.timings
    ]


def integration_error_rows(result):
    return [{"method": m, "max_abs_dx": e} for m, e in result.max_state_error.items()]


## 0. Free-motion showcase

Catalog UR5 under gravity with **zero joint friction and no torque**. Meshcat HTML animation inline.


In [4]:
demo = UR5Manipulator()
demo.params = catalog_params_no_damping()
q0 = np.array([0.0, -np.pi / 2 + 0.2, 0.0, -np.pi / 2, 0.0, 0.0])
v0 = np.array([0.25, -0.15, 0.2, -0.05, 0.1, -0.05])
demo.x0 = demo.q2x(q0, v0)
demo.compute_forced(
    lambda t: np.zeros(demo.m),
    tf=1.5,
    n_steps=45,
    compile_backend="jax",
    verbose=True,
)




===               Time-Domain Simulation                  ===
system: 'UR5 Manipulator'
n=12, m=6
x0: [ 0.       -1.370796  0.       -1.570796] ... (12 values)
interval: [0, 1.5]
n_pts=45, dt=0.0340909
solver: 'scipy' (auto-selected)
compile_backend='jax'
solver_options: {'method': 'RK45', 'rtol': 0.0001, 'atol': 1e-07, 'use_jac': False}
------------------------------------------------------------
Running integration...
Completed in 11.251 seconds
------------------------------------------------------------
n_samples: 45
x_final: [-0.067061  4.557299 -0.583284 -4.32826 ] ... (12 values)
integration_stats: {'solver': 'scipy', 'mode': 'forced', 'nfev': 500, 'njev': 0, 'nlu': 0, 'success': True, 'status': 0}


Trajectory(t=array([0.        , 0.03409091, 0.06818182, 0.10227273, 0.13636364,
       0.17045455, 0.20454545, 0.23863636, 0.27272727, 0.30681818,
       0.34090909, 0.375     , 0.40909091, 0.44318182, 0.47727273,
       0.51136364, 0.54545455, 0.57954545, 0.61363636, 0.64772727,
       0.68181818, 0.71590909, 0.75      , 0.78409091, 0.81818182,
       0.85227273, 0.88636364, 0.92045455, 0.95454545, 0.98863636,
       1.02272727, 1.05681818, 1.09090909, 1.125     , 1.15909091,
       1.19318182, 1.22727273, 1.26136364, 1.29545455, 1.32954545,
       1.36363636, 1.39772727, 1.43181818, 1.46590909, 1.5       ]), x=array([[ 0.00000000e+00,  1.31142251e-02,  3.51854720e-02,
         6.56816741e-02,  1.03766684e-01,  1.48186062e-01,
         1.97095606e-01,  2.47954118e-01,  2.97770735e-01,
         3.43881430e-01,  3.84605274e-01,  4.19216877e-01,
         4.47523096e-01,  4.69356906e-01,  4.84053473e-01,
         4.89799155e-01,  4.82723681e-01,  4.56296210e-01,
         4.04640408e-01,  

In [5]:
demo.animate(renderer="meshcat", is_3d=True, html=True)

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7000/static/


## 1. Manipulator equation of motion

Consider a serial $n$-joint manipulator in generalized coordinates $q \in \mathbb{R}^n$. The standard second-order model is

$$
H(q)\,\ddot q + C(q,\dot q)\,\dot q + d(q,\dot q) + g(q) = \tau,
$$

where $H(q) \in \mathbb{R}^{n \times n}$ is the symmetric positive-definite inertia matrix, $C(q,\dot q)\dot q$ collects Coriolis and centrifugal terms, $g(q)$ is gravity, $d$ is dissipation, and $\tau$ is applied joint torque.

Define the **bias force** (everything that is not inertia times acceleration):

$$
b(q,\dot q) = C(q,\dot q)\,\dot q + g(q).
$$

**Forward dynamics** solves for the generalized acceleration $\ddot q$ given $(q,\dot q,\tau)$:

$$
\ddot q = H(q)^{-1}\bigl(\tau - b(q,\dot q) - d(q,\dot q)\bigr).
$$

For simulation, minilink stacks state $x = [q;\dot q] \in \mathbb{R}^{2n}$ so that

$$
\dot x = \begin{bmatrix} \dot q \\ \ddot q \end{bmatrix} = f(x,u) = \begin{bmatrix} \dot q \\ \text{FD}(q,\dot q,u) \end{bmatrix}.
$$

On the UR5 catalog plant, **`H`**, **`C`**, and **`g`** are built from spatial RNEA; the default **`forward_dynamics`** uses the Articulated-Body Algorithm (ABA). Below we **disable viscous damping** ($d=0$) so the symbolic Lagrange export matches the spatial parameters.


## 2. Pipeline A — Recursive Newton–Euler + explicit $H$ (RNEA–$H$)

### 2.1 Spatial vectors

Each link $i$ carries a 6-vector spatial velocity $v_i = \begin{bmatrix} \omega_i \\ v_i \end{bmatrix}$ and spatial force $f_i$. The **spatial inertia** $I_i$ maps acceleration to force. The **motion cross** operator $\mathrm{crm}(v)$ and its dual $\mathrm{crm}(v)^\top$ appear in the force balance

$$
f_i = I_i a_i - \mathrm{crm}(v_i)^\top I_i v_i.
$$

### 2.2 Inverse dynamics (one tree pass each way)

Given $(q,\dot q,\ddot q)$, RNEA computes $\tau$ in two sweeps:

**Outward** (base $\to$ tip): for each joint $i$,

$$
v_i = {}^{i}\!X_{i-1}\, v_{i-1} + S_i\,\dot q_i, \qquad
a_i = {}^{i}\!X_{i-1}\, a_{i-1} + S_i\,\ddot q_i + \mathrm{crm}(v_i)\,S_i\,\dot q_i,
$$

then accumulate $f_i$ from $I_i$, $a_i$, and $v_i$.

**Inward** (tip $\to$ base): project forces onto joints,

$$
\tau_i = S_i^\top f_i, \qquad f_{i-1} \mathrel{+}= {}^{i}\!X_{i-1}^\top f_i.
$$

We write $\tau = \mathrm{RNEA}(q,\dot q,\ddot q)$.

### 2.3 Building $H$ and forward dynamics

The bias force at zero acceleration is

$$
b(q,\dot q) = \mathrm{RNEA}(q,\dot q, 0).
$$

Each column of the inertia matrix is one inverse-dynamics call with a unit joint acceleration:

$$
H_{:,j}(q) = \mathrm{RNEA}(q, 0, e_j), \qquad j = 1,\ldots,n,
$$

followed by symmetrization $H \leftarrow \tfrac12(H + H^\top)$. Forward dynamics is the linear solve

$$
\ddot q = H^{-1}(\tau - b - d).
$$

**Complexity:** one bias pass $O(n)$, $n$ columns $O(n^2)$, solve $O(n^3)$ — for UR5 ($n=6$) the solve is negligible; forming $H$ dominates.

**minilink:** `UR5Manipulator.forward_dynamics_rnea_h`, and `H` / `g` / `C` via the same spatial RNEA stack.


## 3. Pipeline B — Articulated Body Algorithm (ABA)

ABA computes the **same** $\ddot q$ as RNEA–$H$ but never assembles $H(q)$. It maintains articulated-body inertias $I_i^A$ and bias forces $p_{A,i}$ while propagating along the kinematic tree.

### 3.1 Pass 1 — outward (velocities and bias)

For each link $i$, with joint motion subspace $S_i$ and parent transform ${}^{i}\!X_{i-1}$:

$$
v_i = {}^{i}\!X_{i-1}\, v_{i-1} + S_i\,\dot q_i, \qquad
c_i = \mathrm{crm}(v_i)\,S_i\,\dot q_i,
$$

$$
p_{A,i} = -\mathrm{crm}(v_i)^\top I_i v_i.
$$

### 3.2 Pass 2 — inward (articulated inertia)

Initialize $I_i^A = I_i$. From tip to base, for each $i$:

$$
U_i = I_i^A S_i, \qquad d_i = S_i^\top U_i, \qquad u_i = \tau_i - S_i^\top p_{A,i},
$$

$$
I_i^A \leftarrow I_i^A - \frac{U_i U_i^\top}{d_i}, \qquad
p_{A,i-1} \mathrel{+}= {}^{i}\!X_{i-1}^\top\!\left(p_{A,i} + I_i^A c_i + \frac{U_i u_i}{d_i}\right),
$$

with the articulated inertia $I_i^A$ propagated to the parent before processing the next link.

### 3.3 Pass 3 — outward (accelerations)

Starting from the base spatial acceleration $a_0$ (gravity), for each $i$:

$$
a_i = {}^{i}\!X_{i-1}\, a_{i-1} + c_i, \qquad
\ddot q_i = \frac{u_i - U_i^\top a_i}{d_i}, \qquad
a_i \leftarrow a_i + S_i\,\ddot q_i.
$$

**Complexity:** each pass is $O(n)$, so **$O(n)$ per forward-dynamics call**.

**minilink:** `UR5Manipulator.forward_dynamics` (catalog default for simulation and `f`).


## 4. Pipeline C — Symbolic Lagrange derivation

### 4.1 Energies

Build a DH chain in SymPy. With kinetic energy $T(q,\dot q)$ and potential $V(q)$,

$$
L(q,\dot q) = T(q,\dot q) - V(q).
$$

For rigid links, $T = \sum_i \tfrac12 v_{c,i}^\top M_i v_{c,i} + \tfrac12 \omega_i^\top I_i \omega_i$ expressed in joint coordinates.

### 4.2 Euler–Lagrange equations

$$
\frac{d}{dt}\frac{\partial L}{\partial \dot q} - \frac{\partial L}{\partial q} = \tau.
$$

Expanding the $\dot q$-dependent terms yields the standard manipulator form

$$
H(q)\,\ddot q + C(q,\dot q)\,\dot q + g(q) = \tau,
$$

where

$$
H_{ij} = \frac{\partial^2 T}{\partial \dot q_i \partial \dot q_j}, \qquad
g_i = \frac{\partial V}{\partial q_i},
$$

and the Coriolis matrix $C$ follows from Christoffel symbols of $H(q)$ (or equivalent Kane/Lagrange bookkeeping).

### 4.3 Export and evaluation

SymPy derives $H(q)$, $C(q,\dot q)$, and $g(q)$ symbolically. `to_minilink(backend="jax")` **lambdifies** them once into JAX callables; each forward-dynamics step evaluates $H$ and solves

$$
\ddot q = H^{-1}(\tau - C\dot q - g - d),
$$

so runtime cost is similar to RNEA–$H$, while the **upfront** derive + export cost is large (minutes for UR5).

**minilink:** `minilink.symbolic.mechanics` → `build_symbolic_ur5()` (helpers cell above).


### 4.4 Complexity summary

| Pipeline | Dominant per-step cost | Forms $H(q)$? |
| --- | --- | --- |
| RNEA–$H$ | $O(n^2)$ from $n$ RNEA columns | yes |
| ABA | $O(n)$ three tree passes | no |
| Symbolic Lagrange (numeric) | $O(n^2)$ evaluate $H$ + $O(n^3)$ solve | yes |

All three target the same $\ddot q$ when the underlying model data match; they differ in **algorithm** and **when** work is paid (symbolic: mostly upfront).


## 5. Hands-on — one configuration

Before the batch study, call each pipeline on the same $(q, \dot q, \tau)$.


In [6]:
arm = UR5Manipulator()
params = catalog_params_no_damping()

q = np.array([0.1, -0.5, 0.2, -1.0, 0.3, 0.0])
v = np.array([0.5, -0.3, 0.2, 0.1, -0.4, 0.2])
u = np.zeros(6)

qdd_rnea = forward_dynamics_rnea_h(arm, q, v, u, params)
qdd_aba = forward_dynamics_aba(arm, q, v, u, params)

print("RNEA–H qdd:", np.array2string(qdd_rnea, precision=4, suppress_small=True))
print("ABA    qdd:", np.array2string(qdd_aba, precision=4, suppress_small=True))
print("max |Δqdd| (ABA vs RNEA–H):", np.max(np.abs(qdd_rnea - qdd_aba)))


RNEA–H qdd: [  1.2386  22.9298 -25.3939  27.0263  -4.2004 -23.6337]
ABA    qdd: [  1.2386  22.9298 -25.3939  27.0263  -4.2004 -23.6337]
max |Δqdd| (ABA vs RNEA–H): 4.618527782440651e-14


### Build the symbolic plant (one JAX export)

Copy catalog DH / mass / inertia into SymPy, derive Lagrange $H,C,g$ (`simplify=False` for speed), then **lambdify once** to a JAX plant. That same object is reused for the batch and the EoM integration — do not export a second time.

First call is slow (about a minute); later cells hit the cache. Set `SKIP_SYMBOLIC = True` to run ABA-only.


In [ ]:
SKIP_SYMBOLIC = os.environ.get("UR5_SKIP_SYMBOLIC", "").strip().lower() in {
    "1",
    "true",
    "yes",
}

symbolic_plant = None
if not SKIP_SYMBOLIC:
    try:
        symbolic_plant = build_symbolic_ur5(
            backend=DEFAULT_INTEGRATION_BACKEND, verbose=True
        )
        qdd_sym = forward_dynamics_symbolic(symbolic_plant, q, v, u)
        print("Symbolic qdd:", np.array2string(qdd_sym, precision=4, suppress_small=True))
        print(
            "max |Δqdd| (Symbolic vs RNEA–H):", np.max(np.abs(qdd_rnea - qdd_sym))
        )
    except ImportError:
        print("SymPy not installed — pip install minilink[symbolic]")
else:
    print("Skipping symbolic pipeline.")


Deriving symbolic UR5 EoM (Lagrange, simplify=False)...
  derive: 265.8 s
Exporting to minilink (backend='jax')...


### Symbolic inertia $H(q)$ — size and complexity

Before lambdify, $H$ is an explicit SymPy matrix. Term counts = top-level summands; ops = `sympy.count_ops`.


In [ ]:
if symbolic_plant is None:
    print("Symbolic plant not built — skip complexity cell.")
else:
    sym_sys = derive_symbolic_ur5(verbose=False)
    H_stats = symbolic_matrix_complexity(sym_sys.H)
    C_stats = symbolic_matrix_complexity(sym_sys.C)
    g_stats = symbolic_matrix_complexity(sym_sys.g)

    rows = []
    for name, st in (("H(q)", H_stats), ("C(q, q̇)", C_stats), ("g(q)", g_stats)):
        rows.append(
            {
                "matrix": name,
                "shape": f"{st['shape'][0]}×{st['shape'][1]}",
                "total_terms": st["total_terms"],
                "max_terms_entry": st["max_terms"],
                "mean_terms": f"{st['mean_terms']:.1f}",
                "total_ops": st["total_ops"],
                "zeros": st["n_zero"],
            }
        )
    cols = [
        "matrix",
        "shape",
        "total_terms",
        "max_terms_entry",
        "mean_terms",
        "total_ops",
        "zeros",
    ]
    display(Markdown("#### Aggregate complexity"))
    display(Markdown(rows_to_markdown(rows, cols)))
    display(Markdown("#### Top-level term counts in $H_{ij}$"))
    display(Markdown(format_complexity_table(H_stats)))

    i_d, j_d = H_stats["densest_entry"]
    entry_str = str(sym_sys.H[i_d, j_d])
    display(
        Markdown(
            f"Most-ops entry **$H_{{{i_d + 1},{j_d + 1}}}$**: "
            f"{int(H_stats['term_counts'][i_d, j_d])} terms, "
            f"{int(H_stats['op_counts'][i_d, j_d])} ops, "
            f"string length {len(entry_str):,}."
        )
    )
    print(entry_str[:900] + (" ..." if len(entry_str) > 900 else ""))


## 6. Single-step batch — accuracy and timing

Fast smoke: **5 random** $(q,\dot q,\tau)$ samples (seed 0), **5** timing repeats. One error metric: $\max |\Delta \ddot q|$ vs RNEA–$H$.


In [ ]:
SEED = DEFAULT_SEED
N_SAMPLES = DEFAULT_N_SAMPLES
N_TIMING = DEFAULT_N_TIMING

configs = build_evaluation_batch(seed=SEED, n_samples=N_SAMPLES)
result = comprehensive_comparison(
    arm,
    symbolic_plant,
    configs,
    params=params,
    seed=SEED,
    n_timing=N_TIMING,
)
print(f"Batch: {result.n_samples} random samples (seed={SEED})")

acc_cols = ["method", "max_abs_dqdd", "median_ms"]
display(Markdown("### Accuracy and timing (vs RNEA–H)"))
display(Markdown(rows_to_markdown(accuracy_summary(result), acc_cols)))
display(Markdown(f"**ABA median speedup vs RNEA–H:** {aba_speedup(result):.2f}×"))


## 7. EoM integration — stepping $\dot x = f(x,u)$

Integrate the catalog plants with fixed-step RK4 and compare trajectories. Symbolic is checked in §5–6 only (closed-form $H$ is too slow for a multi-step timer).

| Phase | What it measures |
| --- | --- |
| **compile** | `Simulator` construction |
| **JIT warm-up** | first `solve()` |
| **integrate** | second `solve()` (EoM integration only) |

Smoke grid: $t_f = 0.05\,\mathrm{s}$, $\Delta t = 50\,\mathrm{ms}$ (one step), zero torque, JAX.


In [ ]:
integ_result, ref_traj = eom_integration_comparison(
    params,
    symbolic_plant=None,  # single-step already covers symbolic
    compile_backend=DEFAULT_INTEGRATION_BACKEND,
    tf=DEFAULT_INTEGRATION_TF,
    dt=DEFAULT_INTEGRATION_DT,
)
print(
    f"EoM grid: tf={integ_result.tf}s, dt={integ_result.dt}s, "
    f"n={integ_result.n_steps}, backend={integ_result.backend}"
)

timing_cols = ["method", "compile_ms", "jit_warmup_ms", "integrate_ms", "backend"]
display(Markdown("### EoM integration timings"))
display(Markdown(rows_to_markdown(integration_timing_rows(integ_result), timing_cols)))
if integ_result.max_state_error:
    display(Markdown("### Trajectory error vs RNEA–H"))
    display(
        Markdown(
            rows_to_markdown(
                integration_error_rows(integ_result), ["method", "max_abs_dx"]
            )
        )
    )


## 8. Summary

* **Single-step FD:** ABA matches RNEA–$H$ tightly; symbolic is in the same ballpark but much slower per call.
* **EoM integration:** RNEA–$H$ and ABA trajectories agree on a short JAX grid; JIT warm-up dominates once, then integration is cheap.
* **Speed:** ABA wins because it never forms $H$; symbolic evaluates a large closed-form inertia (best for analysis, not long sims).

**When to use which:** ABA for simulation loops; RNEA–$H$ / symbolic $H$ for teaching, linearization, and control design.
